### DATA INGESTION

In [0]:
# Leitura da tabela.
df = spark.read.csv("/Volumes/workspace/default/retail-demand-intelligence/train.csv", header = True, inferSchema = True)

# Exibir os dados.
display(df)

# Exibir o esquema da tabela
df.printSchema()

In [0]:
# Contagem de linhas.
df.count()

In [0]:
# Contagem de colunas.
len(df.columns)

In [0]:
from pyspark.sql.functions import col, sum, when

# Valores nulos.
df_nulos = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(df_nulos)

In [0]:
from pyspark.sql.functions import min, max

# Intervalo de datas.

df_intervalo = df.select(
    min("Date").alias("data_minima"),
    max("Date").alias("data_maxima")
)

display(df_intervalo)

In [0]:
from pyspark.sql.functions import count

# Contagem de dados duplicados

df_duplicates = df.groupBy(df.columns).agg(count("*").alias("contagem")).filter(col("contagem") > 1)

df_duplicates.show()

### DATA PROFILING

In [0]:
# Quantas lojas existem?
store_number = df.select("Store").distinct().count()
store_number

In [0]:
# Verificar valores de cada variável categórica
print("Valores de Open:")
df.select("Open").distinct().show()
print("Valores de Promo:")
df.select("Promo").distinct().show()
print("Valores de StateHoliday:")
df.select("StateHoliday").distinct().show()
print("Valores de SchoolHoliday:")
df.select("SchoolHoliday").distinct().show()
print("Valores de DayOfWeek:")
df.select("DayOfWeek").distinct().show()


In [0]:
# Estatística para Sales
df.select("Sales").summary("mean", "stddev","min", "25%", "50%", "75%", "max").show()

# Estatística para Custormers
df.select("Customers").summary("mean", "stddev","min", "25%", "50%", "75%", "max").show()


In [0]:
# Contagem de Sales igual a 0.
df.filter(df.Sales == 0).count()

In [0]:
# Contagem de Open igual a 0.
df.filter(df.Open == 0).count()

Podemos afirmar que há 54 registros de diferença entre Sales = 0 e Open = 0. Logo existem dias que não venderam com a loja aberta


In [0]:
# Quantidade dias com a loja aberta sem vendas
df.filter(
    (df.Sales == 0) & (df.Open == 1)
).count()

In [0]:
# Quantidade de vendas com a loja fechada
df.filter(
    (df.Sales > 0) & (df.Open == 0)
).count()

Precisamos verificar se 0 vendas representa um comportamento válido do neegócio ou um problema de qualidade dos dados

In [0]:
#Verificar dados com Sales = 0 e Open = 1
df.filter(
    (df.Sales == 0) & (df.Open == 1)
).select(
    "Store",
    "DayOfWeek",
    "Date",
    "Sales",
    "Customers",
    "Promo",
    "StateHoliday",
    "SchoolHoliday"
).orderBy("Date").show(truncate=False) # Truncate permite que dados longos sejam exibidos na tebala ao invés de somente "...".

In [0]:
df.filter(
    (df.Sales == 0) & (df.Open == 1)
).groupby("Store").count().orderBy("count", ascending=False).show()

Parece ser um comportamento comum e não algo específico de uma loja. Logo devemos manter os 54 registros.


In [0]:
# Contagem de ocorrência de feriados
df.groupBy("StateHoliday").count().orderBy("StateHoliday").show()

In [0]:
# Contagem de ocorrência de lojas abertas/ fechadas por feriados.
df.groupBy("Open", "StateHoliday").count().orderBy("Open", "StateHoliday").show()

In [0]:
from pyspark.sql.functions import date_format

# Verificar se o dia da semana está correto.
df.select(
    # Converter data em dia da semana
    date_format(df.Date, "EEEE").alias("dia_da_semanda"),
    "DayOfWeek"
).distinct().orderBy("DayOfWeek").show()

In [0]:
# Quantos registros existem por loja
df.groupBy("Store").count().orderBy("count").show()

In [0]:
# Verificando se todas as lojas possuem registros do inicio ao fim do intervalo de datas
df.groupBy("Store").agg(
    min("Date").alias("data_inicio"),
    max("Date").alias("data_fim")
).orderBy("data_inicio").show()

In [0]:
# Valdiar a quantidade de registros por loja
df.groupBy("Store").count().groupBy("count").count().orderBy("count").show()

In [0]:
# Valdiar a data por loja
df.groupBy("Store").agg(
    min("Date").alias("data_inicio"),
    max("Date").alias("data_fim")
).groupBy(
    "data_inicio",
    "data_fim"
).count().show()

### Faltam dados para algumas lojas. Por isso, é necessário investigar se a ausência de dados está relacionada a abertura de lojas ou se pode ser um problema no dataset.

In [0]:
# Verificar todas as lojas que possuem 758 registros.
lojas_758 = (
    df.groupBy("Store").count().filter(col("count") == 758).orderBy("Store")
)

lojas_758.show()

In [0]:
# Verificar a contagem de dias abertos e fechados
df.filter(col("Store").isin(
    [row["Store"] for row in lojas_758.collect()])
).groupBy("Open").count().show()

In [0]:
# Verificar as datas de aberturas de cada loja
df.filter(col("Store").isin(
    [row["Store"] for row in lojas_758.collect()])
).groupBy("Store").agg(
    min("Date").alias("data_abertura"),
    max("Date").alias("data_fim")
).orderBy("data_abertura").show(200)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, datediff

lojas_758 = (
    df.groupBy("Store").count().filter(col("count") == 758).select("Store")
)

# Agrupar os dados por loja sem reduzir a uma única linha de resultado
window_store = Window.partitionBy("Store").orderBy("Date")

# Identificar os gaps entre as datas
df_gaps = (
    # Realizar inner join para manter os registros com somente 758.
    df.join(lojas_758, on="Store", how="inner")
    # Lag acessa uma linha anterior ao resultaado atual. (No caso, um dia antes)
    .withColumn("data_anterior", lag(col("Date")).over(window_store))
    # Calcula o intervalo entre a data atual e a última data em que há registro
    .withColumn("dias_desde_anterior", datediff(col("Date"), col("data_anterior")))
)

In [0]:
# Contagem de ocorrência de gaps maiores que 1 dia.
df_gaps.filter(col("dias_desde_anterior") > 1).count()

In [0]:
# Visualizar quais lojas possuem esses gaps e qual o tamanho deles.
df_gaps.filter(
    col("dias_desde_anterior") > 1
).select(
    "Store",
    "Date",
    "data_anterior",
    "dias_desde_anterior"
).orderBy("Store", "Date").show(180)

In [0]:
# Verificar a contagem de datas com a ausência.
df_gaps.filter(
    col("dias_desde_anterior") > 1
).groupBy("data_anterior", "Date", "dias_desde_anterior").count().orderBy("dias_desde_anterior", ascending=False).show()

In [0]:
# Verificar se há somente um registro por dia em cada loja.
df.groupBy("Store", "Date").count().filter(col("count") > 1).count()

#EDA


In [0]:
from pyspark.sql.functions import (
    year,
    month,
    avg,
    sum,
    min,
    max
)

# Vendas por mês
vendas_mensais = (
    df.filter(col("Open") == 1)
    .groupBy(
        year("Date").alias("ano"),
        month("Date").alias("mes")
    )
    .agg(
        avg("Sales").alias("media_vendas"),
        sum("Sales").alias("total_vendas"),
        min("Sales").alias("min_vendas"),
        max("Sales").alias("max_vendas"),
    )
    .orderBy("ano", "mes")
)

vendas_mensais.show(50)

O mês de dezembro se apresenta como um período de vendas elevadas

In [0]:
# Vendas por dia da semana.
vendas_dia_semana = (
    df.filter(col("Open") == 1)
    .groupBy("DayOfWeek")
    .agg(
        avg("Sales").alias("media_vendas"),
        sum("Sales").alias("total_vendas"),
        min("Sales").alias("min_vendas"),
        max("Sales").alias("max_vendas"),
    )
    .orderBy("DayOfWeek")
)

vendas_dia_semana.show()

In [0]:
# Quantidade de dados por dia da semana
df.filter(col("Open") == 1).groupBy("DayOfWeek").count().orderBy("DayOfWeek").show()

Domingo apresenta uma média de vendas elevadas, porém com uma quantidade de vendas menor.


In [0]:
from pyspark.sql.functions import countDistinct

# Lojas abertas por dia na semana
df.filter(col("Open") == 1).groupBy("DayOfWeek").agg(countDistinct("Store")).alias("loajs_abertas").orderBy("DayOfWeek").show()

A baixa quantidade de vendas no domingo está relacionado a quantidade pequena de lojas abertas nesse período. Mesmo com um número menor de lojas abertas, o domingo apresenta excelentes indicadores.

In [0]:
# Verificando a influência das promoções no resultado final.
vendas_promo = (df.filter(col("Open") == 1)
                .groupBy("Promo")
                .agg(
                    avg("Sales").alias("media_vendas"),
                    sum("Sales").alias("total_vendas"),
                    count("*").alias("quantidade_vendas")
                )
                .orderBy("Promo")
                )

vendas_promo.show()

In [0]:
listCorr = ["Sales", "Customers", "Open", "Promo", "SchoolHoliday", "DayOfWeek"]

for i in listCorr:
    for j in listCorr:
        if i != j:
            corr = df.stat.corr(i, j)
            print(f"Correlação entre {i} e {j}: {corr}")

In [0]:
# Verificar média de vendas com ou sem promoção ao longo da semana
df.filter(col("Open") == 1).groupBy("DayOfWeek", "Promo").agg(
    avg("Sales").alias("media_vendas"),
    count("*").alias("qtde_lojas")
).orderBy("DayOfWeek").show()

In [0]:
# Verificar indicadoresd de vendas por loja.
df.filter(col("Open") == 1).groupBy("Store").agg(
    avg("Sales").alias("media_vendas"),
    sum("Sales").alias("total_vendas"),
    min("Sales").alias("min_vendas"),
    max("Sales").alias("max_vendas"),
    count("Sales").alias("qtde_vendas")
).orderBy("total_vendas").show(200)

In [0]:
vendas_loja = (
    df.filter(col("Open") == 1)
    .groupBy("Store")
    .agg(
        avg("Sales").alias("media_vendas"),
        count("*").alias("quantidade_observadas")
    )
    .orderBy(col("media_vendas").desc())
)

vendas_loja.show(180)

In [0]:
from pyspark.sql.functions import median, stddev

df.filter(col("Open") == 1).groupBy("Store").agg(
    avg("Sales").alias("media_vendas"),
    min("Sales").alias("min_vendas"),
    max("Sales").alias("max_vendas"),
    median("Sales").alias("mediana_vendas"),
    stddev("Sales").alias("desvio_vendas")
).orderBy("max_vendas", ascending=False).show(180)

In [0]:
vendas_loja.agg(
    avg("media_vendas").alias("media"),
    median("media_vendas").alias("mediana"),
    stddev("media_vendas").alias("desvio_padrao"),
    min("media_vendas").alias("min"),
    max("media_vendas").alias("max")
).show()

Percebemos que a maior média de um loja pode ser até 8 vezes que a média geral. Também é observado que a média geral é maior que a mediana, indicando que pode haver dados grandes que influenciam a média. 

In [0]:
# Top 10 lojas com maior média em vendas
vendas_loja.select(
    "Store",
    "media_vendas",
    ).orderBy(
        col("media_vendas").desc()
    ).show(10)

In [0]:
# Bottom 10 lojas com maior média em vendas.
vendas_loja.select(
    "Store",
    "media_vendas"
    ).orderBy(
        col("media_vendas").asc()
    ).show(10)

Store, Date, Promo são features relevantes para explicar a demanda. Custormers também apresenta forte associação, mas pode ser uma informação indisponível no momento da previsão.

# Feature Engineering

In [0]:
from pyspark.sql.functions import dayofmonth, weekofyear

df_features = (
    df
    .withColumn("ano", year(col("Date")))
    .withColumn("mes", month(col("Date")))
    .withColumn("dia", dayofmonth(col("Date")))
    .withColumn("semana", weekofyear(col("Date")))
)

df_features.select(
    "Date",
    "ano",
    "mes",
    "dia",
    "semana"
).show(10)

In [0]:
# Aplicando feature histórica de lag_1 (dia anterior).

df_features = (df_features.withColumn(
    "Sales_lag_1",
    lag(col("Sales")).over(Window.partitionBy("Store").orderBy("Date"))
))

df_features.show(10)

In [0]:
# Não está pegando a venda do dia anterior e sim a última observação.
df_features.filter((col("Store") == 13) & (col("Date") >= "2013-01-01") & (col("Date") <= "2015-01-05")).select(
    "Store",
    "Date",
    "Sales",
    "Sales_lag_1"
).orderBy("Date", ascending=False).show(50)

In [0]:
# Cria a coluna com a data anterior.
df_features = (
    df_features
    .withColumn(
        "Date_lag_1",
        lag("Date").over(
            Window.partitionBy("Store").orderBy("Date")
        )
    )
)

# Cria uma coluna contando a diferença de dias entre a Data e a Data Anterior
df_features = (
    df_features
    .withColumn(
        "dias_desde_anterior",
        datediff(col("Date"), col("Date_lag_1"))
    )
)

# Não está pegando a venda do dia anterior e sim a última observação.
df_features.filter((col("Store") == 13) & (col("Date") >= "2013-01-01") & (col("Date") <= "2015-01-05")).select(
    "Store",
    "Sales",
    "Sales_lag_1",
    "Date",
    "Date_lag_1",
    "dias_desde_anterior"
).orderBy("Date", ascending=False).show(50)

In [0]:
from pyspark.sql.functions import when

# Só vamos aplicar o Sales_lag_1 quanto o dias_desde_anterior for == 1, caso contrário, coloque NULL.
df_features = (
    df_features.withColumn(
        "Sales_lag_1",
        when(
            col("dias_desde_anterior") == 1,
            col("Sales_lag_1")
        ).otherwise(None)
    )
)

df_features.filter((col("Store") == 13) & (col("Date") >= "2013-01-01") & (col("Date") <= "2015-01-05")).select(
    "Store",
    "Sales",
    "Sales_lag_1",
    "Date",
    "Date_lag_1",
    "dias_desde_anterior"
).orderBy("Date", ascending=False).show(50)

In [0]:
# Verificando quantas observações terão NULL.
df_features.filter(col("Sales_lag_1").isNull() | (col("dias_desde_anterior") != 1)).count()

In [0]:
# Contagem de quantos dias anterior diferente de 1
df_features.filter(
    col("dias_desde_anterior") != 1
).groupBy(
    "dias_desde_anterior"
).count().orderBy(
    "dias_desde_anterior"
).show()

In [0]:
# Lag de 7 observações
df_features = (
    df_features.withColumn(
        "Date_lag_7",
        lag("Date", 7).over(
            Window.partitionBy("Store").orderBy("Date")
        )
    )
)

# Calculo da diferença de dias entre a Data e a Data Anterior
df_features = (
    df_features.withColumn(
        "dias_desde_lag_7",
        datediff(col("Date"), col("Date_lag_7"))
    )
)

df_features.filter(
    (col("Store") == 13) &
    (col("Date") >= "2014-06-25") &
    (col("Date") <= "2015-01-10")
).select(
    "Store",
    "Date",
    "Date_lag_7",
    "dias_desde_lag_7"
).orderBy("Date").show(30)

In [0]:
# Define vendas para quando o dia anterior for igual a 7
df_features = (
    df_features
    .withColumn(
        "Sales_lag_7",
        when(
            col("dias_desde_lag_7") == 7,
            lag("Sales", 7).over(
                Window.partitionBy("Store").orderBy("Date")
            )
        ).otherwise(None)
    )
)

df_features.filter(
    (col("Store") == 13) &
    (col("Date") >= "2014-06-25") &
    (col("Date") <= "2015-01-15")
).select(
    "Store",
    "Date",
    "Sales",
    "Sales_lag_7",
    "Date_lag_7",
    "dias_desde_lag_7"
).orderBy("Date").show(30)

In [0]:
# Organize os dados em intervalos de 7 dias
window_7d = (
    Window
    .partitionBy("Store")
    .orderBy(col("Date").cast("timestamp").cast("long"))
    .rangeBetween(-7 * 86400, -86400)
)

In [0]:
# Criar coluna com a média móvel de 7 dias
df_features = (
    df_features.withColumn(
        "Sales_rolling_7",
        avg("Sales").over(window_7d)
    )
)

In [0]:
# Vamos contar a quantidade de registros dentro da média móvel
df_features = (
    df_features
    .withColumn(
        "qtd_registros_7d",
        count("Sales").over(window_7d)
    )
)

df_features.filter(
    (col("Store") == 13) &
    (col("Date") >= "2014-06-25") &
    (col("Date") <= "2015-01-15")
).select(
    "Store",
    "Date",
    "Sales",
    "qtd_registros_7d"
).orderBy("Date").show(30)

In [0]:
# modificar a coluna de méddia móvel de 7 dias somente quando houver 7 registros.
df_features = (
    df_features
    .withColumn(
        "Sales_rolling_7",
        when(
            col("qtd_registros_7d") == 7,
            avg("Sales").over(window_7d)
        ).otherwise(None)
    )
)

df_features.filter(
    (col("Store") == 13) &
    (col("Date") >= "2014-06-25") &
    (col("Date") <= "2015-01-15")
).select(
    "Store",
    "Date",
    "Sales",
    "qtd_registros_7d",
    "Sales_rolling_7"
).orderBy("Date").show(30)

In [0]:
# Quantificar NULLs no dataset.
df_features.select(
    when(col("Sales_lag_1").isNull(), "NULL")
        .otherwise("Preenchido")
        .alias("lag_1"),

    when(col("Sales_lag_7").isNull(), "NULL")
        .otherwise("Preenchido")
        .alias("lag_7"),

    when(col("Sales_rolling_7").isNull(), "NULL")
        .otherwise("Preenchido")
        .alias("rolling_7")
).groupBy(
    "lag_1",
    "lag_7",
    "rolling_7"
).count().show()

In [0]:
# Removendo valores nulos.
df_model = df_features.dropna(
    subset=["Sales_lag_1", "Sales_lag_7", "Sales_rolling_7"]
)

df_model.count()

In [0]:
df_model.select(
    "Store",
    "Date",
    "Sales",
    "Sales_lag_1",
    "Sales_lag_7",
    "Sales_rolling_7"
).show(10)

In [0]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

# Todas as colunas categóricas.
categorical_cols = [
    "Store",
    "DayOfWeek",
    "ano",
    "mes",
    "dia",
    "semana"
]

# Para cada coluna in categorical_cols, crie uma StringIndexer.
indexers = [
    # StringIndexer tansforma valores das colunas categóricas em índices númericos. Atribuindo os menores índices às menores categorias menos frequentes.
    StringIndexer(
        inputCol = coluna,
        outputCol = f"{coluna}_index"
    )

    for coluna in categorical_cols
]

# Criando uma pipeline para executar todas as transformações.
pipeline_indexers = Pipeline(stages=indexers)

# Treina o modelo da pipeline para aprender realizar as transformações.
model_indexers = pipeline_indexers.fit(df_model) # Estimator

# Pega o dataset atual e aplica as transformações aprendidas
df_indexed = model_indexers.transform(df_model)


In [0]:
df_indexed.select(
    "Store",
    "Store_index",
    "DayOfWeek",
    "DayOfWeek_index",
    "ano",
    "ano_index",
    "mes",
    "mes_index",
    "dia",
    "dia_index",
    "semana",
    "semana_index"
).show(20)

In [0]:
df_indexed.select(
    "DayOfWeek",
    "DayOfWeek_index"
).distinct().orderBy("DayOfWeek").show()

In [0]:
df_indexed.select(
    "mes",
    "mes_index"
).distinct().orderBy("mes").show()

In [0]:
# OneHotEncoder transforma os índices numéricos em um vetor binário para o modelo interpretar como uma categoria.
from pyspark.ml.feature import OneHotEncoder

encoders = [
    OneHotEncoder(
        inputCol = f"{coluna}_index",
        outputCol = f"{coluna}_encoded"
    )
    for coluna in categorical_cols
]

pipeline_encoders = Pipeline(stages=encoders)

model_encoders = pipeline_encoders.fit(df_indexed)

df_encoded = model_encoders.transform(df_indexed)


In [0]:
df_encoded.select(
    "Store",
    "Store_index",
    "Store_encoded",
    "DayOfWeek",
    "DayOfWeek_index",
    "DayOfWeek_encoded",
    "ano",
    "ano_index",
    "ano_encoded",
    "mes",
    "mes_index",
    "mes_encoded",
    "dia",
    "dia_index",
    "dia_encoded",
    "semana",
    "semana_index",
    "semana_encoded"
).show(20)

In [0]:
# Converter todas as colunas em uma única chamada features.
from pyspark.ml.feature import VectorAssembler

encoded_cols = [
    "Store_encoded",
    "DayOfWeek_encoded",
    "ano_encoded",
    "mes_encoded",
    "dia_encoded",
    "semana_encoded"
]

numeric_cols = [
    "Promo",
    "SchoolHoliday",
    "Open",
    "Sales_lag_1",
    "Sales_lag_7",
    "Sales_rolling_7"
]

feature_cols = encoded_cols + numeric_cols

# Criando o VectorAssembler
assembler = VectorAssembler(
    inputCols = feature_cols,
    outputCol = "features"
)

# Transformando em DataFrame
df_assembled = assembler.transform(df_encoded)

df_assembled.select(
    "Store",
    "Date",
    "Sales",
    "features"
).show(5, truncate = False)

In [0]:
# Divisão de dataset para treino, validação e teste.
df_train = (
    df_model.filter(col("Date") <= "2014-12-31")
)

df_validation = (
    df_model
    .filter(
        (col("Date") >= "2015-01-01") &
        (col("Date") <= "2015-05-31")
    )
)

df_test = (
    df_model.filter(col("Date") >= "2015-06-01")
)

In [0]:
print("Treino:", df_train.count())
print("Validação:", df_validation.count())
print("Teste:", df_test.count())

In [0]:
df_train.select("Date").agg(
    min("Date").alias("data_min"),
    max("Date").alias("data_max")
).show()

df_validation.select("Date").agg(
    min("Date").alias("data_min"),
    max("Date").alias("data_max")
).show()

df_test.select("Date").agg(
    min("Date").alias("data_min"),
    max("Date").alias("data_max")
).show()

O correto é dividir os dados de treino, validação e teste antes de realizar os treinamentos ou qualquer função com fit. Por isso vamos refazer o treinamento.

In [0]:
categorical_cols = [
    "Store",
    "DayOfWeek",
    "ano",
    "mes",
    "dia",
    "semana"
]

indexers = [
    StringIndexer(
        inputCol=coluna,
        outputCol=f"{coluna}_index",
        handleInvalid="keep"
    )
    for coluna in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{coluna}_index",
        outputCol=f"{coluna}_encoded"
    )
    for coluna in categorical_cols
]

encoded_cols = [
    f"{coluna}_encoded"
    for coluna in categorical_cols
]

numeric_cols = [
    "Promo",
    "SchoolHoliday",
    "Open",
    "Sales_lag_1",
    "Sales_lag_7",
    "Sales_rolling_7"
]

feature_cols = encoded_cols + numeric_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

feature_pipeline = Pipeline(
    stages=indexers + encoders + [assembler]
)

feature_model = feature_pipeline.fit(df_train)

In [0]:
# Transformando os dados em df
df_train_prepared = feature_model.transform(df_train)

df_validation_prepared = feature_model.transform(df_validation)

df_test_prepared = feature_model.transform(df_test)

In [0]:
# Separação dos dados para o treinamento.
df_train_ml = df_train_prepared.select(
    "features",
    "Sales"
)

df_validation_ml = df_validation_prepared.select(
    "features",
    "Sales"
)

df_test_ml = df_test_prepared.select(
    "features",
    "Sales"
)

# Model Training


In [0]:
from pyspark.ml.regression import LinearRegression

# Criação do modelo
lr = LinearRegression(
    featuresCol = "features",
    labelCol = "Sales"
)

# Treinamento
lr_model = lr.fit(df_train_ml)

# Validaçaõ
lr_predictions = lr_model.transform(df_validation_ml)

lr_predictions.select(
    "Sales",
    "prediction"
).show(10)

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

# Calcular o RMSE, MAE e R²

# Mesmo que o MAE, porém com penalidade maiores para erros maiores
evaluator_rmse = RegressionEvaluator(
    labelCol = "Sales",
    predictionCol = "prediction",
    metricName = "rmse"
)

# Em média, quantas unidades de vendas o modelo erra
evaluator_mae = RegressionEvaluator(
    labelCol = "Sales",
    predictionCol = "prediction",
    metricName = "mae"
)

# Indica o quanto o modelo acerta com os dados
evaluator_r2 = RegressionEvaluator(
    labelCol = "Sales",
    predictionCol = "prediction",
    metricName = "r2"
)

In [0]:
rmse = evaluator_rmse.evaluate(lr_predictions)
mae = evaluator_mae.evaluate(lr_predictions)
r2 = evaluator_r2.evaluate(lr_predictions)

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R2: {r2}")

Houveram previsões negativas em vendas com valor 0, vamos investigar

In [0]:
qtd_negativas = (
    lr_predictions
    .filter(col("prediction") < 0)
    .count()
)

total_previsoes = lr_predictions.count()

print(f"Previsões negativas: {qtd_negativas}")
print(f"Total de previsões: {total_previsoes}")
print(f"Percentual: {qtd_negativas / total_previsoes * 100:.2f}%")

In [0]:
lr_predictions.filter(
    col("prediction") < 0
).select("Sales", "prediction").orderBy("prediction").show(20)

Modelo: Linear Regression

Validação temporal:
2015-01-01 → 2015-05-31

RMSE: 1499.10
MAE: 1092.62
R²: 0.8477

Previsões negativas:
17.683 / 167.105
10,58%

In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol = "features",
    labelCol = "Sales",
    numTrees = 100,
    seed = 42
)

rf_model = rf.fit(df_train_ml)

rf_predictions = rf_model.transform(df_validation_ml)

rf_predictions.select(
    "Sales",
    "prediction"
).show(10)

In [0]:
# Parâmetros de avaliação.
rmse_rf = evaluator_rmse.evaluate(rf_predictions)
mae_rf = evaluator_mae.evaluate(rf_predictions)
r2_rf = evaluator_r2.evaluate(rf_predictions)

print(f"RMSE: {rmse_rf}")
print(f"MAE: {mae_rf}")
print(f"R2: {r2_rf}")

RandomForest apresenta melhoria para parte significativa do dataset, mas erra em valores maiores.

In [0]:
qtd_negativas_rf = (
    rf_predictions
    .filter(col("prediction") < 0)
    .count()
)

total_previsoes_rf = rf_predictions.count()

print(f"Previsões negativas: {qtd_negativas_rf}")
print(f"Total de previsões: {total_previsoes_rf}")
print(
    f"Percentual: "
    f"{qtd_negativas_rf / total_previsoes_rf * 100:.2f}%"
)

%md
Modelo: Random Forest
trees: 100
seed: 42

Validação temporal:
2015-01-01 → 2015-05-31

RMSE: 1509.88
MAE: 996.85
R²: 0.8455

Previsões negativas:
0%

Vamos alterar os hiperparâmetros para avaliar a melhoria do modelo

In [0]:
# Treinamento com 50 árvores.
rf_50 = RandomForestRegressor(
    featuresCol="features",
    labelCol="Sales",
    numTrees=50,
    seed=42
)

rf_50_model = rf_50.fit(df_train_ml)

rf_50_predictions = rf_50_model.transform(df_validation_ml)

In [0]:
rmse_rf_50 = evaluator_rmse.evaluate(rf_50_predictions)
mae_rf_50 = evaluator_mae.evaluate(rf_50_predictions)
r2_rf_50 = evaluator_r2.evaluate(rf_50_predictions)

print(f"RMSE: {rmse_rf_50:.2f}")
print(f"MAE: {mae_rf_50:.2f}")
print(f"R²: {r2_rf_50:.4f}")

%md
%md
Modelo: Random Forest
trees: 50
seed: 42

Validação temporal:
2015-01-01 → 2015-05-31

RMSE: 1502.68
MAE: 985.78
R²: 0.8469

Previsões negativas:
0%

In [0]:
rf_200 = RandomForestRegressor(
    featuresCol="features",
    labelCol="Sales",
    numTrees=200,
    seed=42
)

rf_200_model = rf_200.fit(df_train_ml)

rf_200_predictions = rf_200_model.transform(df_validation_ml)

In [0]:
rmse_rf_200 = evaluator_rmse.evaluate(rf_200_predictions)
mae_rf_200 = evaluator_mae.evaluate(rf_200_predictions)
r2_rf_200 = evaluator_r2.evaluate(rf_200_predictions)

print(f"RMSE: {rmse_rf_200:.2f}")
print(f"MAE: {mae_rf_200:.2f}")
print(f"R²: {r2_rf_200:.4f}")

Modelo: Random Forest
trees: 200
seed: 42

Validação temporal:
2015-01-01 → 2015-05-31

RMSE: 1517.43
MAE: 999.15
R²: 0.8439

Previsões negativas:
0%

In [0]:
# Vamos testar o algoritmo com maxDepth, que indica até onde uma árvore pode crescer.
rf_depth_5 = RandomForestRegressor(
    featuresCol="features",
    labelCol="Sales",
    numTrees=50,
    maxDepth=5,
    seed=42
)

rf_depth_5_model = rf_depth_5.fit(df_train_ml)

rf_depth_5_predictions = rf_depth_5_model.transform(
    df_validation_ml
)

In [0]:
rmse_depth_5 = evaluator_rmse.evaluate(rf_depth_5_predictions)
mae_depth_5 = evaluator_mae.evaluate(rf_depth_5_predictions)
r2_depth_5 = evaluator_r2.evaluate(rf_depth_5_predictions)

print(f"RMSE: {rmse_depth_5:.2f}")
print(f"MAE: {mae_depth_5:.2f}")
print(f"R²: {r2_depth_5:.4f}")

In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf_depth_10 = RandomForestRegressor(
    featuresCol="features",
    labelCol="Sales",
    numTrees=50,
    maxDepth=10,
    seed=42
)

rf_depth_10_model = rf_depth_10.fit(df_train_ml)

rf_depth_10_predictions = rf_depth_10_model.transform(
    df_validation_ml
)

In [0]:
rmse_depth_10 = evaluator_rmse.evaluate(rf_depth_10_predictions)
mae_depth_10 = evaluator_mae.evaluate(rf_depth_10_predictions)
r2_depth_10 = evaluator_r2.evaluate(rf_depth_10_predictions)

print(f"RMSE: {rmse_depth_10:.2f}")
print(f"MAE: {mae_depth_10:.2f}")
print(f"R²: {r2_depth_10:.4f}")

Modelo: Random Forest trees: 50 seed: 42 maxdepth: 10

Validação temporal: 2015-01-01 → 2015-05-31

RMSE: 1112.43 MAE: 697.11 R²: 0.9161

Previsões negativas: 0%

Vamos automatizar as avaliações e os parâmetros com MLflow

In [0]:
dbutils.fs.mkdirs(
    "/Volumes/workspace/default/retail-demand-intelligence/mlflow_tmp"
)

In [0]:
# Cálculo das métricas.
rmse_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="r2"
)

rmse = rmse_evaluator.evaluate(rf_depth_10_predictions)
mae = mae_evaluator.evaluate(rf_depth_10_predictions)
r2 = r2_evaluator.evaluate(rf_depth_10_predictions)

In [0]:
# # Criar uma execução do experimento e registrar seus resultados.
import mlflow

with mlflow.start_run(run_name = "random_forest_depth_10") as run:
    
    # Registro dos parâmetros.
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("seed", 42)

    # Registros das métricas
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # Registro do modelo.
    mlflow.spark.log_model(
        rf_depth_10_model,
        "model"
    )

    print("Run ID:", run.info.run_id)

In [0]:
import mlflow



with mlflow.start_run(run_name="random_forest_depth_10_model") as run:

    # Registro dos parâmetros.
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("seed", 42)

    # Registro das métricas.
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # Registro do modelo Spark ML.
    mlflow.spark.log_model(
        rf_depth_10_model,
        "model",
        dfs_tmpdir="/Volumes/workspace/default/retail-demand-intelligence/mlflow_tmp"
    )

    print("Run ID:", run.info.run_id)

In [0]:
# Gerar previsões no conjunto de teste.
rf_test_predictions = rf_depth_10_model.transform(df_test_ml)

# Calcular métricas no conjunto de teste.
rmse_test = rmse_evaluator.evaluate(rf_test_predictions)
mae_test = mae_evaluator.evaluate(rf_test_predictions)
r2_test = r2_evaluator.evaluate(rf_test_predictions)

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"R²: {r2_test:.4f}")

O Random Forest com 50 árvores, profundidade máxima 10 e seed 42 apresentou RMSE de 1.043,12, MAE de 659,35 e R² de 0,9247 no conjunto de teste. O desempenho foi ligeiramente superior ao observado na validação (RMSE 1.112,43; MAE 697,11; R² 0,9161), não indicando degradação de generalização entre os dois períodos. A avaliação foi realizada sobre dados posteriores ao período utilizado para treinamento e seleção dos hiperparâmetros.

In [0]:
from pyspark.sql.functions import abs

df_erros = (
    rf_test_predictions
    .withColumn("erro", col("Sales") - col("prediction"))
    .withColumn("erro_absoluto", abs(col("Sales") - col("prediction")))
    .withColumn("erro_percentual", when(col("Sales") != 0, abs(col("Sales") - col("prediction")) / col("Sales") * 100)))


display(
    df_erros.select(
        "Sales",
        "prediction",
        "erro",
        "erro_absoluto",
        "erro_percentual"
    )
    .orderBy(
        col("erro_absoluto").desc()
    )
    .limit(20)
)

O modelo possui dificuldades em prever valores elevados

# Inferência

In [0]:
import mlflow

model_uri = "runs:/6baf42d8aa6647b9aa51174fb4b43830/model"

loaded_model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/workspace/default/retail-demand-intelligence/mlflow_tmp"
)

print("Modelo carregado com sucesso.")

In [0]:
# Gerar previsões utilizando o modelo carregado no MLflow.
loaded_model_predictions = loaded_model.transform(df_test_ml)

display(
    loaded_model_predictions.select(
        "Sales",
        "prediction"
    ).limit(20)
)

In [0]:
from pyspark.ml import PipelineModel

# Cria um modelo de Pipeline de dados com o modelo de RandomForest
full_model = PipelineModel(
    stages=feature_model.stages + [rf_depth_10_model]
)

# Teste do Modelo
full_predictions = full_model.transform(
    df_test
)

display(
    full_predictions.select(
        "Store",
        "Date",
        "Sales",
        "prediction"
    ).limit(20)
)

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="r2"
)

rmse = rmse_evaluator.evaluate(full_predictions)
mae = mae_evaluator.evaluate(full_predictions)
r2 = r2_evaluator.evaluate(full_predictions)

print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

In [0]:
# Registrar no MLflow o artefato.
import mlflow

with mlflow.start_run(run_name="retail_demand_full_pipeline") as run:

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("seed", 42)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    mlflow.spark.log_model(
        full_model,
        "model",
        dfs_tmpdir="/Volumes/workspace/default/retail-demand-intelligence/mlflow_tmp"
    )

    print("Run ID:", run.info.run_id)

In [0]:
full_predictions.select("prediction").printSchema()

In [0]:
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec

output_schema = Schema([
    ColSpec("double", "prediction")
])

signature = ModelSignature(
    inputs=signature.inputs,
    outputs=output_schema
)

print(signature)

In [0]:
from mlflow.models import infer_signature

input_example_spark = df_test.limit(5)

input_example = input_example_spark.toPandas()

output_example = full_model.transform(
    input_example_spark
)

# Criando schema para a criação da sinatura do modelo.
signature = infer_signature(
    input_example,
    output_example.toPandas()
)

print(signature)

In [0]:
with mlflow.start_run(run_name="retail_demand_full_pipeline_signature") as run:

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("seed", 42)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # Agora são adicionadas a assinatura do modelo e um exemplo de entrada.
    mlflow.spark.log_model(
        full_model,
        "model",
        signature=signature,
        input_example=input_example,
        dfs_tmpdir="/Volumes/workspace/default/retail-demand-intelligence/mlflow_tmp"
    )

    print("Run ID:", run.info.run_id)

In [0]:
# Predição a partir do artefato.
model_uri = f"runs:/520421a5d2914749b961db218cf60724/model"

loaded_full_model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/workspace/default/retail-demand-intelligence/mlflow_tmp"
)

final_predictions = loaded_full_model.transform(df_test)

display(
    final_predictions.select(
        "Store",
        "Date",
        "Sales",
        "prediction"
    ).limit(20)
)

In [0]:
run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"

# Registrando modelo no Unity Catalog
registered_model_name = "workspace.default.retail_demand_model"

mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(
    "name='workspace.default.retail_demand_model'"
)

for version in versions:
    print(
        f"Modelo: {version.name}\n"
        f"Versão: {version.version}\n"
        f"Run ID: {version.run_id}\n"
    )

In [0]:
registered_model_uri = (
    "models:/workspace.default.retail_demand_model/1"
)

registered_model = mlflow.spark.load_model(
    registered_model_uri,
    dfs_tmpdir="/Volumes/workspace/default/retail-demand-intelligence/mlflow_tmp"
)

registered_predictions = registered_model.transform(df_test)

display(
    registered_predictions.select(
        "Store",
        "Date",
        "Sales",
        "prediction"
    ).limit(20)
)


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="Sales",
    predictionCol="prediction",
    metricName="r2"
)

final_rmse = rmse_evaluator.evaluate(registered_predictions)
final_mae = mae_evaluator.evaluate(registered_predictions)
final_r2 = r2_evaluator.evaluate(registered_predictions)

print(f"RMSE: {final_rmse:.2f}")
print(f"MAE: {final_mae:.2f}")
print(f"R²: {final_r2:.4f}")